In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l3.2 Scaled dot-product

The core formula is `softmax(QK^T / sqrt(d)) V`. Build it by hand, then check it
against the engine primitive.

In [ ]:
from torch.nn import functional as F
from pocketlm import scaled_dot_product_attention

torch.manual_seed(0)
q = torch.randn(1, 4, 16)
k = torch.randn(1, 4, 16)
v = torch.randn(1, 4, 16)
att = (q @ k.transpose(-2, -1)) / (16 ** 0.5)
w_hand = F.softmax(att, dim=-1)
out_hand = w_hand @ v
out, w = scaled_dot_product_attention(q, k, v)
print("weights match:", torch.allclose(w, w_hand))
print("output match: ", torch.allclose(out, out_hand))

Why divide by sqrt(d)? Without it, dot products grow with dimension, the
softmax saturates to near one-hot, and gradients vanish. The scale keeps
attention smooth and trainable.

In [ ]:
assert torch.allclose(w, w_hand)
assert torch.allclose(out, out_hand)